### Structured OP 

### pydantic

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.3-70b-versatile")
res = model.invoke("why do fish swim?")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10c26c2f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10c26cd70>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
  title:str = Field(description="The title of the movie")
  year:int = Field(description="The release date of the movie")
  director:str = Field(description="The director of the movie")
  rating:float = Field(description="The rating of the movie out of 10")

In [10]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10c26c2f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10c26cd70>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The release date of the movie', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie out 

In [5]:
model.invoke("Provide me the details of the movie Fight Club")

AIMessage(content="**Fight Club (1999) - A Psychological Thriller**\n\nFight Club is a thought-provoking, dark, and subversive film directed by David Fincher, based on the 1996 novel of the same name by Chuck Palahniuk. The movie explores themes of toxic masculinity, consumerism, and the search for meaning in a postmodern society.\n\n**Plot**\n\nThe film follows an unnamed narrator (played by Edward Norton), a white-collar worker suffering from insomnia and a sense of purposelessness. He attends support groups for testicular cancer survivors, where he meets Marla Singer (played by Helena Bonham Carter), a mysterious and troubled woman.\n\nThe narrator's life takes a dramatic turn when he meets Tyler Durden (played by Brad Pitt), a charismatic and unconventional soap maker. The two men form a fight club, where they engage in brutal and cathartic battles with other men. The fight club becomes a sensation, attracting disillusioned men from all walks of life.\n\nAs the fight club grows in 

In [11]:
model_with_structure.invoke("Provide me the details of the movie Fight Club")

Movie(title='Fight Club', year=1999, director='David Fincher', rating=8.8)

### Message output along with parsed structure

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9twvjy4sj', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 290, 'total_tokens': 322, 'completion_time': 0.057622852, 'completion_tokens_details': None, 'prompt_time': 0.014103701, 'prompt_tokens_details': None, 'queue_time': 0.050855074, 'total_time': 0.071726553}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7062-d918-7280-96b3-e3891dfd1514-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': '9twvjy4sj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 32,

### Nested structure

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
  name: str
  role: str

class MovieDetails(BaseModel):
  title: str
  year: int
  cast: list[Actor]
  genres: list[str]
  budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie scarface")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')], genres=['Action', 'Sci-Fi'], budget=160.0)

### type DICT

In [ ]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie Batman Arkham Night")
response

{'budget': 100000000,
 'director': 'Sefton Hill',
 'rating': 9,
 'title': 'Batman: Arkham Knight',
 'year': 2015}

In [20]:
class Actor(TypedDict):
  name: str
  role: str

class MovieDetails(TypedDict):
  title: str
  year: int
  cast: list[Actor]
  genres: list[str]
  budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie scarface")
response

{'budget': 25,
 'cast': [{'name': 'Al Pacino', 'role': 'Tony Montana'},
  {'name': 'Michelle Pfeiffer', 'role': 'Elvira Hancock'}],
 'genres': ['Crime', 'Drama'],
 'title': 'Scarface',
 'year': 1983}

In [21]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

### Data classes

In [23]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-4o-mini",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='5e1f1d75-3550-4b86-b8c2-f594be79a5ce'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 125, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_24787597d7', 'id': 'chatcmpl-DkcMj7HeDlpEPsTzWrtA66q93GJJA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7071-532d-7ae1-9bc6-654b06a5d866-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens'

In [25]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [ ]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [26]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-4o-mini",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')